In [2]:
"""
PIPELINE STAGE: JSON Metadata Extraction & Relational Spatial Transformation
LIBRARIES: os, pandas, json

1. OBJECTIVE:
   Extract geospatial intelligence (Latitude, Longitude) from unstructured or semi-structured 
   JSON objects. This stage flattens array-based or dictionary-based spatial data into a 
   strict relational format, preparing the geographic coordinates necessary for subsequent 
   regional energy clustering and spatial mapping analysis.

2. FILE ARCHITECTURE:
   - Input (Geo JSON): os.path.join("..", "..", "raw_data", "geo_metadata", "geo_lat_lon.json")
   - Output (Cleaned Geo): os.path.join("..", "..", "processed_data", "steps", "10_a_geo_metadata_cleaned.xlsx")
   - System: Dynamically generates the output directory structure if it does not exist.

3. POLYGLOT EXTRACTION ARCHITECTURE (Type-Aware Parsing):
   - Dynamic Schema Handling: Implements an 'Instance Check' to autonomously handle both 
     Sequential Lists (Array of Objects) and Associative Dictionaries (Key-Value pairs), 
     ensuring robust data ingestion regardless of the underlying JSON structure.
   - Multi-Key Token Mapping: Utilizes a fallback extraction strategy to combat metadata 
     naming inconsistencies and maintain high recall rates:
       * Plate mappings: Scans for 'plate' OR 'id' OR 'plaka'
       * Coordinate mappings: Scans for 'lat'/'latitude' AND 'lon'/'longitude'

4. DATA NORMALIZATION & VALIDATION:
   - Structural Flattening: Transforms nested hierarchical JSON structures into a flat, 
     long-format pandas DataFrame.
   - Type Enforcement: Enforces explicit numerical integrity by casting the 'Plate' identifier to an Integer.
   - Sorting Logic: Strictly sorts the final DataFrame in ascending order by 'Plate' (1-81).

5. METHODOLOGICAL SIGNIFICANCE:
   By decoupling geospatial extraction from the main pipeline and isolating it into a 
   dedicated Excel asset, the architecture allows for an independent spatial audit before 
   merging coordinates into the longitudinal 2020-2024 master energy database.
"""

import pandas as pd
import json
import os

def json_kordinat_ayristirma_liste():
    # Dosya Yolları
    JSON_PATH = os.path.join("..", "..", "raw_data", "geo_metadata", "geo_lat_lon.json")
    OUT_PATH = os.path.join("..", "..", "processed_data", "steps", "10_a_geo_metadata_cleaned.xlsx")
    
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    
    try:
        print(f"JSON dosyası okunuyor: {JSON_PATH}")
        with open(JSON_PATH, 'r', encoding='utf-8') as f:
            geo_data = json.load(f)
        
        geo_list = []
        
        # HATA ÇÖZÜMÜ: Eğer veri bir liste ise direkt üzerinde dönüyoruz
        if isinstance(geo_data, list):
            for item in geo_data:
                # Plaka bilgisini 'plate' veya 'id' anahtarından almaya çalışalım
                plate = item.get('plate') or item.get('id') or item.get('plaka')
                lat = item.get('lat') or item.get('latitude')
                lon = item.get('lon') or item.get('longitude')
                
                if plate is not None:
                    geo_list.append({
                        'Plate': int(plate),
                        'Latitude': lat,
                        'Longitude': lon
                    })
        
        # Eğer veri hala sözlük ise (tedbir amaçlı)
        elif isinstance(geo_data, dict):
            for plate, coords in geo_data.items():
                lat = coords.get('lat') or coords.get('latitude')
                lon = coords.get('lon') or coords.get('longitude')
                geo_list.append({
                    'Plate': int(plate),
                    'Latitude': lat,
                    'Longitude': lon
                })

        # DataFrame oluştur
        df_geo = pd.DataFrame(geo_list)
        
        if df_geo.empty:
            print("UYARI: JSON içinden veri çekilemedi. Anahtar isimlerini (lat, lon, plate) kontrol edin.")
            return

        # Plaka numarasına göre sırala
        df_geo = df_geo.sort_values(by='Plate', ascending=True)
        
        # Excel'e kaydet
        df_geo.to_excel(OUT_PATH, index=False)
        
        print("\n" + "="*40)
        print("BAŞARILI: Liste formatındaki JSON Excel'e dönüştürüldü.")
        print(f"Toplam Kayıt: {len(df_geo)}")
        print(f"Kayıt Yeri: {OUT_PATH}")
        print("="*40)

    except Exception as e:
        print(f"HATA: {e}")

if __name__ == "__main__":
    json_kordinat_ayristirma_liste()

JSON dosyası okunuyor: ..\..\raw_data\geo_metadata\geo_lat_lon.json

BAŞARILI: Liste formatındaki JSON Excel'e dönüştürüldü.
Toplam Kayıt: 81
Kayıt Yeri: ..\..\processed_data\steps\10_a_geo_metadata_cleaned.xlsx
